# CardioSafe AI - Notebook 09: the combined module (ECG finding x medication)

Builds `cardiosafe_infer.py` (loads your trained ensemble + QT estimator and analyses an ECG) and `cardiosafe_combine.py` (the rules that link ECG findings to prescribed drugs).

**Three outcomes per (ECG finding, drug), always with the quoted label sentence:**
1. **relationship found** - the FDA label says the drug can cause the finding, or cautions/contraindicates it in patients who have the finding
2. **none found in available sources** - label retrieved, nothing about the finding (never means "safe")
3. **insufficient evidence** - drug not recognised, no label, only weak mentions, or the ECG estimate itself is unreliable

Sections: (1) check the engine reproduces your earlier ECG results, (2) check the rules on 13 scenarios with the real drug labels, (3) full report on real test ECGs.
Runtime: GPU recommended (T4); about 5 minutes. Needs notebooks 03 to 08 done.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'; KB = WORK + '/kb'
sys.path.insert(0, WORK)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_infer.py
"""CardioSafe ECG inference engine: 15-model calibrated ensemble + conformal sets + learned QT/RR estimator.
Loads everything produced by notebooks 02-06b from the project folder. Works on CPU."""
import json, os
import numpy as np
import torch

import cardiosafe_common as C
import cardiosafe_models as MZ
import cardiosafe_qtnet as QN
import cardiosafe_trust as T


def qt_status(qtc, sex, band, male=450.0, female=460.0):
    """Three-way rule validated in notebook 06b: prolonged if estimate - band >= limit; normal if estimate + band < limit."""
    thr = male if int(sex) == 0 else female
    if qtc - band >= thr:
        return 'prolonged', thr
    if qtc + band < thr:
        return 'normal', thr
    return 'borderline', thr


def prepare_signal(sig, fs=100, n_samples=1000):
    """-> (1000, 12) float32 at 100 Hz. Accepts (T, 12) or (12, T); resamples if needed. Raises ValueError on a bad shape."""
    a = np.asarray(sig, dtype=np.float64)
    if a.ndim != 2 or 12 not in a.shape:
        raise ValueError(f'expected a 12-lead array, got shape {a.shape}')
    if a.shape[1] != 12:
        a = a.T
    if fs != 100:
        from scipy.signal import resample
        a = resample(a, int(round(a.shape[0] * 100 / fs)), axis=0)
    if a.shape[0] < n_samples:
        raise ValueError(f'need at least 10 s at 100 Hz ({n_samples} samples), got {a.shape[0]}')
    a = a[:n_samples]
    if not np.isfinite(a).all():
        raise ValueError('signal contains NaN/inf')
    return a.astype(np.float32)


class ECGEngine:
    def __init__(self, work, device=None):
        self.work = work
        self.dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.trust = json.load(open(f'{work}/trust_config.json'))
        art = np.load(f'{work}/trust_artifacts.npz')
        self.Tcal, self.ths = float(art['T']), np.asarray(art['thresholds'])
        self.cal = {(c, lab): art[f'cal_{c}_{lab}'] for c in range(5) for lab in (0, 1)}
        ns = np.load(f'{work}/norm_stats.npz'); self.mu, self.sd = ns['mu'], ns['sd']
        self.qcfg = json.load(open(f'{work}/qtc_config_learned.json'))
        self.classes = self.trust['classes']
        self.members = []
        for name in self.trust['members']:
            net = MZ.REGISTRY[name.rsplit('_s', 1)[0]]().to(self.dev)
            net.load_state_dict(torch.load(f'{work}/models/{name}.pt', map_location=self.dev)); net.eval()
            self.members.append(net)
        self.qnets = []
        for ck in self.qcfg['checkpoints']:
            net = MZ.XResNet1D(n_classes=2, width=32).to(self.dev)
            net.load_state_dict(torch.load(f'{work}/{ck}', map_location=self.dev)); net.eval()
            self.qnets.append(net)
        self.mu_y, self.sd_y = np.array(self.qcfg['target_mean']), np.array(self.qcfg['target_sd'])

    def analyze(self, X, sex, alpha=0.1):
        """X: (N, 1000, 12) (or one record (1000, 12)); sex: int or array (0 male, 1 female). Returns a list of dicts."""
        X = np.asarray(X, dtype=np.float32)
        if X.ndim == 2:
            X = X[None]
        sex = np.broadcast_to(np.asarray(sex), (len(X),))
        Xt = C.to_tensor_ncl(X, self.mu, self.sd, self.dev)
        P = np.stack([C.predict(m, Xt) for m in self.members])                  # (members, N, 5)
        p = T.apply_T(P.mean(0), self.Tcal)
        dis = T.disagreement_score(P)
        oc = T.conformal_outcome(T.conformal_pvalues(self.cal, p), alpha)         # 0 neg, 1 pos, 2 uncertain, 3 empty
        Q = np.mean([QN.predict_regress(n, Xt, self.mu_y, self.sd_y) for n in self.qnets], 0)
        qsd = np.std([QN.predict_regress(n, Xt, self.mu_y, self.sd_y)[:, 0] for n in self.qnets], 0)
        out = []
        for i in range(len(X)):
            qt, rr = float(Q[i, 0]), float(Q[i, 1])
            qtc = qt / np.cbrt(rr / 1000.0)
            status, thr = qt_status(qtc, sex[i], self.qcfg['borderline_band_ms'], self.qcfg['male_threshold_ms'], self.qcfg['female_threshold_ms'])
            out.append({
                'probs': {c: float(p[i, k]) for k, c in enumerate(self.classes)},
                'predicted': [c for k, c in enumerate(self.classes) if p[i, k] >= self.ths[k]],
                'conformal': {c: ['confident_negative', 'confident_positive', 'uncertain', 'empty'][int(oc[i, k])] for k, c in enumerate(self.classes)},
                'disagreement': float(dis[i]),
                'qt_ms': qt, 'rr_ms': rr, 'hr_bpm': 60000.0 / rr, 'qtc_fridericia_ms': float(qtc), 'qt_seed_sd_ms': float(qsd[i]),
                'qtc_status': status, 'qtc_limit_ms': thr, 'qtc_band_ms': self.qcfg['borderline_band_ms'], 'sex': int(sex[i]),
            })
        return out


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_combine.py
"""Combined module: ECG findings x prescription -> evidence-backed relations with exactly three outcomes per (finding, drug):
  relationship_found       the FDA label text states the finding as an effect of the drug, or cautions/contraindicates the drug
                           in patients who have the finding (sentence, label id and link are attached)
  none_found               label retrieved, no statement about this finding in the available labels (NOT evidence of safety)
  insufficient_evidence    drug not recognised, no label retrieved, only weak mentions, or the ECG finding itself is unreliable
Nothing here is a diagnosis or a causal claim; every statement quotes its source."""
import cardiosafe_kb as K

STRONG = ('strong', 'moderate')
TOPIC_LABEL = {'qt': 'QT prolongation', 'bradycardia': 'slow heart rate (bradycardia)', 'av_block': 'conduction disturbance / heart block'}


def ecg_findings(ecg, hr_cut=50.0, hr_borderline=60.0, qt_unstable_sd_ms=25.0):
    """ecg: dict from ECGEngine.analyze. -> {topic: {'level': present|borderline|absent|unreliable, 'detail': str}}"""
    f = {}
    qtc, st = ecg['qtc_fridericia_ms'], ecg['qtc_status']
    base = f"QTc (Fridericia) estimated {qtc:.0f} ms (limit {ecg['qtc_limit_ms']:.0f} ms, uncertainty band +/-{ecg['qtc_band_ms']:.0f} ms)"
    if ecg.get('qt_seed_sd_ms', 0) > qt_unstable_sd_ms:
        f['qt'] = {'level': 'unreliable', 'detail': base + f"; the three QT models disagree by {ecg['qt_seed_sd_ms']:.0f} ms (SD), so the estimate is not trusted"}
    else:
        f['qt'] = {'level': {'prolonged': 'present', 'borderline': 'borderline', 'normal': 'absent'}[st], 'detail': base}
    hr = ecg['hr_bpm']
    lvl = 'present' if hr < hr_cut else 'borderline' if hr < hr_borderline else 'absent'
    f['bradycardia'] = {'level': lvl, 'detail': f'heart rate estimated {hr:.0f} bpm from the learned RR estimate'}
    cd = ecg['conformal'].get('CD', 'confident_negative')
    lvl = 'present' if cd == 'confident_positive' else 'borderline' if cd in ('uncertain', 'empty') and ecg['probs'].get('CD', 0) >= 0.3 else 'absent'
    f['av_block'] = {'level': lvl, 'detail': f"ECG classifier conduction-disturbance class p={ecg['probs'].get('CD', 0):.2f} (this class includes bundle-branch blocks; "
                                             'AV block / PR interval is not measured separately)'}
    return f


def _cite(e):
    return {'text': e['text'], 'section': e['group'], 'set_id': e.get('set_id'), 'url': e.get('url'), 'brand': (e.get('brand') or [''])[0] if isinstance(e.get('brand'), list) else e.get('brand')}


def assess(ecg, meds, fetch, ddi=None):
    """ecg: ECGEngine result; meds: extract_prescription output; fetch(ingredient) -> label record or None; ddi: optional K.DDI."""
    findings = ecg_findings(ecg)
    rows, notes, ingredients, nolabel = [], [], [], set()
    for m in meds:
        if m['status'] != 'ok' or not m['ingredients']:
            rows.append({'drug': m['name'], 'ingredient': None, 'topic': None, 'finding_level': None, 'outcome': 'insufficient_evidence',
                         'reason': f"drug name '{m['name']}' not recognised, so no label could be checked (status: {m['status']})", 'evidence': []})
            continue
        for ing in m['ingredients']:
            if ing not in ingredients:
                ingredients.append(ing)
    recs = {ing: fetch(ing) for ing in ingredients}
    for ing in ingredients:
        for topic, fd in findings.items():
            s = K.summarize_evidence(recs[ing], topic)
            eff, pc = s['by_role']['drug_effect'], s['by_role']['patient_condition']
            if fd['level'] in ('absent', 'unreliable'):
                if fd['level'] == 'absent' and (eff['tier'] in STRONG):
                    notes.append({'ingredient': ing, 'topic': topic, 'note': f"label lists {TOPIC_LABEL[topic]} as a possible effect (tier {eff['tier']}, "
                                  f"{eff['n_positive']}/{s['n_labels']} labels); no matching ECG finding in this tracing", 'evidence': [_cite(e) for e in eff['evidence'][:1]]})
                if fd['level'] == 'unreliable':
                    rows.append({'drug': ing, 'ingredient': ing, 'topic': topic, 'finding_level': 'unreliable', 'outcome': 'insufficient_evidence',
                                 'reason': 'the ECG-derived finding is unreliable: ' + fd['detail'], 'evidence': []})
                continue
            base = {'drug': ing, 'ingredient': ing, 'topic': topic, 'finding_level': fd['level'], 'finding_detail': fd['detail']}
            if s['tier'] == 'no_label':
                if ing not in nolabel:      # one row per ingredient, not one per finding
                    nolabel.add(ing)
                    rows.append({'drug': ing, 'ingredient': ing, 'topic': None, 'finding_level': None, 'outcome': 'insufficient_evidence',
                                 'reason': 'no FDA label could be retrieved for this ingredient, so the ECG findings above could not be checked against it', 'evidence': []})
                continue
            rels = []
            if eff['tier'] in STRONG:
                rels.append({'relation': 'drug_effect', 'tier': eff['tier'], 'n': eff['n_positive'], 'text': f"label lists {TOPIC_LABEL[topic]} as a possible effect of {ing}",
                             'evidence': [_cite(e) for e in eff['evidence']]})
            if pc['tier'] in STRONG:
                rels.append({'relation': 'patient_condition', 'tier': pc['tier'], 'n': pc['n_positive'],
                             'text': f"label advises caution or lists a contraindication for {ing} in patients with {TOPIC_LABEL[topic]}", 'evidence': [_cite(e) for e in pc['evidence']]})
            if rels:
                rows.append({**base, 'outcome': 'relationship_found', 'relations': rels, 'n_labels': s['n_labels'], 'evidence': [c for r in rels for c in r['evidence']],
                             'caveat': ('ECG finding is borderline: ' if fd['level'] == 'borderline' else '') + 'label statements show prominence in the label, not proof that this drug caused this ECG finding'})
            elif max(eff['tier'], pc['tier'], key=lambda t: K.TIER_RANK[t]) == 'weak' or s['tier'] in ('mention', 'weak'):
                rows.append({**base, 'outcome': 'insufficient_evidence', 'reason': 'only weak mentions found (overdose/pharmacology text or a bare mention)', 'evidence': [_cite(e) for e in s['evidence'][:1]]})
            else:
                rows.append({**base, 'outcome': 'none_found', 'n_labels': s['n_labels'],
                             'reason': f"no statement about {TOPIC_LABEL[topic]} in the {s['n_labels']} FDA label(s) retrieved; this is not evidence that the drug is safe", 'evidence': []})
    pairs = []
    qt_found = [r['ingredient'] for r in rows if r['topic'] == 'qt' and r['outcome'] == 'relationship_found' and any(x['relation'] == 'drug_effect' for x in r['relations'])]
    if findings['qt']['level'] in ('present', 'borderline') and len(set(qt_found)) >= 2:
        pairs.append({'type': 'multiple_qt_drugs', 'drugs': sorted(set(qt_found)), 'text': 'two or more prescribed drugs have QT prolongation statements in their labels'})
    if ddi is not None and len(ddi) and len(ingredients) >= 2:
        for r in ddi.check(ingredients):
            pairs.append({'type': 'ddinter', 'drugs': [r['drug_a'], r['drug_b']], 'level': r['level'], 'text': f"DDInter lists a {r['level']} interaction"})
    found = [r for r in rows if r['outcome'] == 'relationship_found']
    n_unrec = sum(r['outcome'] == 'insufficient_evidence' for r in rows)
    overall = 'attention' if found or pairs else 'insufficient' if n_unrec and not any(r['outcome'] == 'none_found' for r in rows) else 'none_found'
    return {'findings': findings, 'rows': rows, 'pairs': pairs, 'background_notes': notes, 'overall': overall}


DISCLAIMER = ('Research prototype for education. It is not a medical device and must not be used for diagnosis or to change treatment. '
              'ECG values are model estimates trained on PTB-XL (single dataset, no external validation); label statements come from FDA drug labels '
              'and show what a label says, not that a drug caused an ECG finding. "None found" never means "safe".')


def render_report(ecg, meds, result):
    """Plain-text/markdown report for the demo."""
    L = ['## ECG', f"Classifier (calibrated, conformal at 90%): " + ', '.join(f"{c} {ecg['probs'][c]:.2f} ({ecg['conformal'][c].replace('_', ' ')})" for c in ecg['probs']),
         f"QT {ecg['qt_ms']:.0f} ms, QTc (Fridericia) {ecg['qtc_fridericia_ms']:.0f} ms -> **{ecg['qtc_status']}** (limit {ecg['qtc_limit_ms']:.0f} ms, band +/-{ecg['qtc_band_ms']:.0f} ms); heart rate {ecg['hr_bpm']:.0f} bpm",
         '']
    if not meds:
        fnd = ecg_findings(ecg)
        L += ['## Findings relevant to medication safety'] + [f"- {TOPIC_LABEL[t]}: **{v['level']}** - {v['detail']}" for t, v in fnd.items()]
        L += ['', '_No prescription provided, so no medication check was done._', '', '> ' + DISCLAIMER]
        return '\n'.join(L)
    L.append('## Medications read from the prescription')
    for m in meds:
        tag = ', '.join(m['ingredients']) if m['status'] == 'ok' else f"UNRECOGNISED ({m['status']})"
        L.append(f"- {m['raw']}  ->  **{tag}**")
    L += ['', '## ECG finding x medication']
    titles = {'attention': 'Relationships found in drug labels - review recommended', 'none_found': 'No label relationship found for the findings present (not proof of safety)',
              'insufficient': 'Insufficient evidence for some or all items'}
    L.append(f"**{titles[result['overall']]}**")
    for r in result['rows']:
        head = f"- **{r['drug']}**" + (f" / {TOPIC_LABEL[r['topic']]} (ECG finding {r['finding_level']})" if r.get('topic') else '')
        if r['outcome'] == 'relationship_found':
            L.append(head + ': **relationship found**')
            for rel in r['relations']:
                L.append(f"    - {rel['text']} (label tier {rel['tier']}, in {rel['n']}/{r['n_labels']} labels)")
                for e in rel['evidence'][:1]:
                    L.append(f"      > \"{e['text'][:300]}\" [{e['section']}]" + (f" ({e['url']})" if e.get('url') else ''))
            L.append('    - ' + r['caveat'])
        elif r['outcome'] == 'none_found':
            L.append(head + ': none found in available sources - ' + r['reason'])
        else:
            L.append(head + ': **insufficient evidence** - ' + r['reason'])
    for p in result['pairs']:
        L.append(f"- Combination: {' + '.join(p['drugs'])}: {p['text']}")
    if result['background_notes']:
        L += ['', '## Other label notes (no matching ECG finding today)']
        L += [f"- {n['ingredient']}: {n['note']}" for n in result['background_notes']]
    L += ['', '> ' + DISCLAIMER]
    return '\n'.join(L)


def drug_only_report(meds, fetch):
    """Prescription-only mode (no ECG): what the FDA labels say about QT, slow heart rate and heart block for each drug."""
    L = ['## Medications read from the prescription']
    for m in meds:
        tag = ', '.join(m['ingredients']) if m['status'] == 'ok' else f"UNRECOGNISED ({m['status']}) - cannot be checked"
        L.append(f"- {m['raw']}  ->  **{tag}**")
    L += ['', '## What the FDA labels say about cardiac effects']
    seen = set()
    for m in meds:
        if m['status'] != 'ok':
            continue
        for ing in m['ingredients']:
            if ing in seen:
                continue
            seen.add(ing); rec = fetch(ing)
            if rec is None or not rec.get('labels'):
                L.append(f"- **{ing}**: insufficient evidence - no FDA label could be retrieved"); continue
            parts = []
            for topic in K.TOPICS:
                s = K.summarize_evidence(rec, topic)
                eff, pc = s['by_role']['drug_effect'], s['by_role']['patient_condition']
                if eff['tier'] in STRONG:
                    e = eff['evidence'][0]
                    parts.append(f"  - {TOPIC_LABEL[topic]}: label lists it as a possible effect (tier {eff['tier']}, {eff['n_positive']}/{s['n_labels']} labels)\n    > \"{e['text'][:260]}\"" + (f" ({e['url']})" if e.get('url') else ''))
                if pc['tier'] in STRONG:
                    e = pc['evidence'][0]
                    parts.append(f"  - {TOPIC_LABEL[topic]}: label advises caution/contraindication in patients who have it (tier {pc['tier']}, {pc['n_positive']}/{s['n_labels']} labels)\n    > \"{e['text'][:260]}\"" + (f" ({e['url']})" if e.get('url') else ''))
            L.append(f"- **{ing}**" + (':' if parts else ': none found in the available FDA labels for QT, slow heart rate or heart block (not evidence of safety)'))
            L += parts
    L += ['', '> ' + DISCLAIMER]
    return '\n'.join(L)


## 1. Load the engine and verify it reproduces notebook 04 / 06b
The calibrated ensemble's test AUROC must match the earlier ensemble (about 0.934; temperature scaling does not change AUROC) and the QTc estimates must match `qtc_test_predictions.csv`. If either differs, something is mis-loaded.

In [ ]:
import importlib, json, numpy as np, pandas as pd, torch
import cardiosafe_common as C, cardiosafe_infer as I, cardiosafe_combine as CB, cardiosafe_kb as K
for m in (I, CB, K): importlib.reload(m)
X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
sex = meta['sex']; ids = meta['ecg_id']
E = I.ECGEngine(WORK)
print('device:', E.dev, '| ensemble members:', len(E.members), '| QT nets:', len(E.qnets))
t_idx = np.where(te)[0]
res = []
for i in range(0, len(t_idx), 256):
    res += E.analyze(X[t_idx[i:i + 256]], sex[t_idx[i:i + 256]])
P = np.array([[r['probs'][c] for c in C.CLASSES] for r in res])
print('test AUROC of the packaged engine:', round(C.macro_auc(Y[te], P), 4), '(notebook 04 ensemble: about 0.934)')
ref = pd.read_csv(WORK + '/qtc_test_predictions.csv')
mine = pd.DataFrame({'ecg_id': ids[te], 'qtc': [r['qtc_fridericia_ms'] for r in res]}).merge(ref, on='ecg_id')
print('max |QTc difference| vs notebook 06b:', round(float((mine.qtc - mine.qtc_fri_hat).abs().max()), 3), 'ms (should be about 0)')

In [ ]:
df = pd.DataFrame(res)
print('QTc status on the test set:', df.qtc_status.value_counts().to_dict())
print('heart rate (bpm) percentiles 1/5/50/95/99:', np.percentile(df.hr_bpm, [1, 5, 50, 95, 99]).round(0))
print('share with HR < 50:', round(float((df.hr_bpm < 50).mean()), 3), '| QT seed SD ms percentiles 50/95/99:', np.percentile(df.qt_seed_sd_ms, [50, 95, 99]).round(1))
cd = pd.DataFrame([r['conformal'] for r in res]); print('conformal outcome per class (share):'); print(cd.apply(lambda c: c.value_counts(normalize=True)).fillna(0).round(3).to_string())

## 2. Rule check on 13 scenarios with the REAL drug labels
These check that the combination logic does what it claims (they are not a clinical validation). Labels are read from your cache or fetched from openFDA.

In [ ]:
API_KEY = os.environ.get('OPENFDA_API_KEY')
try:
    from google.colab import userdata; API_KEY = userdata.get('OPENFDA_API_KEY') or API_KEY
except Exception: pass
fda_cache = K.JsonCache(KB + '/cache_openfda.json')
fetch = lambda ing: K.fda_fetch(ing, fda_cache, API_KEY)

def fake_ecg(qtc=420, hr=72, sex=0, cd='confident_negative', pcd=0.05, sd=5):
    rr = 60000 / hr; thr = 450 if sex == 0 else 460
    st = 'prolonged' if qtc - 28 >= thr else 'normal' if qtc + 28 < thr else 'borderline'
    return {'probs': {'NORM': .5, 'MI': .05, 'STTC': .05, 'CD': pcd, 'HYP': .05}, 'conformal': {'NORM': 'confident_positive', 'MI': 'confident_negative', 'STTC': 'confident_negative', 'CD': cd, 'HYP': 'confident_negative'},
            'qt_ms': qtc * (rr / 1000) ** (1 / 3), 'rr_ms': rr, 'hr_bpm': hr, 'qtc_fridericia_ms': qtc, 'qt_seed_sd_ms': sd, 'qtc_status': st, 'qtc_limit_ms': thr, 'qtc_band_ms': 28.0, 'sex': sex}
def med(name, ings, status='ok'): return {'raw': name, 'name': name, 'status': status, 'ingredients': ings}
SC = [('prolonged QTc + azithromycin', fake_ecg(520), [med('Azithral', ['azithromycin'])], {('azithromycin', 'qt'): 'relationship_found'}),
      ('prolonged QTc + metformin', fake_ecg(520), [med('Glycomet', ['metformin'])], {('metformin', 'qt'): 'none_found'}),
      ('prolonged QTc + unrecognised drug', fake_ecg(520), [med('Xyzzyflam', [], 'no_candidate')], {('Xyzzyflam', None): 'insufficient_evidence'}),
      ('borderline QTc + ondansetron', fake_ecg(465), [med('Zofran', ['ondansetron'])], {('ondansetron', 'qt'): 'relationship_found'}),
      ('normal ECG + sotalol (no finding, only a note)', fake_ecg(420), [med('Betapace', ['sotalol'])], {}),
      ('HR 42 + metoprolol', fake_ecg(420, 42), [med('Metolar', ['metoprolol'])], {('metoprolol', 'bradycardia'): 'relationship_found'}),
      ('conduction disturbance + atenolol', fake_ecg(420, 70, cd='confident_positive', pcd=.9), [med('Aten', ['atenolol'])], {('atenolol', 'av_block'): 'relationship_found'}),
      ('prolonged QTc + two QT drugs', fake_ecg(520), [med('a', ['azithromycin']), med('b', ['ondansetron'])], {('azithromycin', 'qt'): 'relationship_found', ('ondansetron', 'qt'): 'relationship_found'}),
      ('prolonged QTc + paracetamol (Dolo)', fake_ecg(520), [med('Dolo', ['acetaminophen'])], {('acetaminophen', 'qt'): 'none_found'}),
      ('unstable QT estimate + azithromycin', fake_ecg(520, sd=40), [med('Azee', ['azithromycin'])], {('azithromycin', 'qt'): 'insufficient_evidence'}),
      ('prolonged QTc + chlorpromazine (LABEL GAP, documented limitation)', fake_ecg(520), [med('Largactil', ['chlorpromazine'])], {('chlorpromazine', 'qt'): 'none_found'}),
      ('prolonged QTc + ingredient with no label', fake_ecg(520), [med('Ghost', ['ghostdrugxyz'])], {('ghostdrugxyz', None): 'insufficient_evidence'}),
      ('all normal + atorvastatin', fake_ecg(420), [med('Storvas', ['atorvastatin'])], {})]
rows = []
for name, e, meds, exp in SC:
    r = CB.assess(e, meds, fetch)
    got = {(x['drug'], x['topic']): x['outcome'] for x in r['rows']}
    ok = all(got.get(k) == v for k, v in exp.items()) and not {k for k in got if k not in exp}
    rows.append({'scenario': name, 'pass': ok, 'overall': r['overall'], 'combo_alerts': len(r['pairs']), 'notes': len(r['background_notes']), 'got': got if not ok else ''})
fda_cache.save()
sc = pd.DataFrame(rows); print(sc.to_string(index=False)); print('\nscenarios passed:', int(sc['pass'].sum()), '/', len(sc))
sc.drop(columns='got').to_csv(WORK + '/results_09_scenarios.csv', index=False)

## 3. Full report on real test ECGs
One ECG for each QTc status from the test set, each paired with a realistic prescription. Read these like a reviewer: is every claim backed by a quoted sentence?

In [ ]:
import cardiosafe_rx as R
bm = K.load_brand_map(KB + '/brand_map.csv'); rx_cache = K.JsonCache(KB + '/cache_rxnorm.json')
resolve = lambda n: K.resolve_drug(n, bm, rx_cache)
RX = {'prolonged': 'Rx\n1. Tab Azithral 500 mg OD x 5 days\n2. Tab Ondem 4 mg SOS\n3. Tab Dolo 650 mg TDS x 3 days',
      'borderline': 'Rx\n1. Tab Ciplox 500 mg BD x 7 days\n2. Tab Metolar 50 mg OD\n3. Tab Pantocid DSR 1-0-0',
      'normal': 'Rx\n1. Tab Storvas 10 mg HS\n2. Tab Glycomet 500 mg BD\n3. Tab Telma 40 mg OD'}
for status in ('prolonged', 'borderline', 'normal'):
    cand = [k for k, r in enumerate(res) if r['qtc_status'] == status and r['qt_seed_sd_ms'] < 15]
    if not cand: print('no test ECG with status', status); continue
    k = cand[0]; e = res[k]
    meds = R.extract_prescription(RX[status], resolve); result = CB.assess(e, meds, fetch)
    print('=' * 100); print(f"test ECG id {ids[t_idx[k]]}  | true superclass labels: {[c for c, v in zip(C.CLASSES, Y[t_idx[k]]) if v]}")
    print(CB.render_report(e, meds, result))
rx_cache.save(); fda_cache.save()

In [ ]:
json.dump({'qt_unstable_sd_ms': 25, 'hr_bradycardia_bpm': 50, 'hr_borderline_bpm': 60, 'relation_tiers': ['strong', 'moderate'], 'conformal_alpha': 0.1,
           'scenarios_passed': int(sc['pass'].sum()), 'scenarios_total': len(sc)}, open(WORK + '/combine_config.json', 'w'), indent=1)
print('saved combine_config.json')

## Next
Tell me *ran fine* plus: the AUROC / QTc-difference lines from section 1, the scenario table from section 2, and the **prolonged** report from section 3 (select and copy the text). Then notebook 10 builds the live demo.